# Inferencia sobre Hold-Out Set y Extracción de Predicciones

Este cuaderno tiene un objetivo estrictamente operativo: cargar el modelo preentrenado (`RoBERTa`) desde el entorno de producción (Google Drive) y ejecutar una pasada de inferencia sobre el conjunto de datos de evaluación (`test_set_v7.parquet`).

No se realizará ningún ajuste de pesos ni cálculo de gradientes. Extraeremos las probabilidades puras (Softmax) para exportarlas a un CSV local. Este archivo será la base para el análisis visual y la matriz de confusión en el siguiente cuaderno.

In [ ]:
# Celda 1: Aprovisionamiento y Carga del Modelo Preentrenado
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from google.colab import drive

# 1. Montaje del sistema de archivos en la nube
drive.mount('/content/drive')

# 2. Definición de rutas absolutas
BASE_DIR = '/content/drive/MyDrive/MasterEvolve/Proyecto TFM/SITOR'
TEST_DATA_PATH = os.path.join(BASE_DIR, 'data/gold/test_set_v7.parquet')
MODEL_PATH = os.path.join(BASE_DIR, 'modelo/produccion_roberta')

# 3. Ingesta del dataset de evaluación (Hold-Out ciego)
print("Cargando dataset de evaluación...")
df_test = pd.read_parquet(TEST_DATA_PATH, engine='fastparquet')
df_test['full_text'] = df_test['full_text'].fillna("").astype(str)

# Nota: No reajustamos el LabelEncoder. El mapeo id2label ya está integrado en el config.json del modelo.

# 4. Descongelación del modelo en la memoria de vídeo (VRAM)
print("Instanciando Tokenizador y Modelo desde almacenamiento local...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

# 5. Bloqueo de gradientes y envío a GPU
model.eval()  # Modo evaluación estricto
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"Arquitectura cargada correctamente. Modo inferencia activado en: {device}")